In [1]:
from ragas import evaluate
print("done")

c:\Users\manju\Documents\claudecode\documind-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


done


In [2]:
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
print("metrics done")


metrics done


In [3]:
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic
print("LLM wrappers done")

LLM wrappers done


In [5]:
from dotenv import load_dotenv
load_dotenv()
print("env loaded")

env loaded


In [6]:
import os

llm = LangchainLLMWrapper(ChatAnthropic(
    model="claude-sonnet-4-6",
    api_key=os.environ["ANTHROPIC_API_KEY"]
))

print("LLM ready:", llm)

LLM ready: LangchainLLMWrapper(langchain_llm=ChatAnthropic(...))


In [7]:
from datasets import Dataset

# Sample golden dataset - replace these with real Q&A from your DocuMind documents
sample_data = {
    "question": [
        "What is DocuMind AI?",
        "What embedding model does DocuMind use?",
        "How does DocuMind handle document ingestion?",
        "What search strategy does DocuMind use?",
        "What is the reranking model used in DocuMind?"
    ],
    "answer": [
        "DocuMind AI is a production multi-modal RAG system that enables intelligent document querying using hybrid search and large language models.",
        "DocuMind uses the sentence-transformers/all-MiniLM-L6-v2 model loaded via HuggingFace transformers for generating embeddings.",
        "DocuMind uses Unstructured.io in hi_res mode to parse and chunk documents before storing them in Weaviate.",
        "DocuMind uses BM25 and dense vector hybrid search with Reciprocal Rank Fusion (RRF) to combine results.",
        "DocuMind uses BAAI/bge-reranker-v2-m3 as a cross-encoder reranker to rerank retrieved chunks before passing to the LLM."
    ],
    "contexts": [
        ["DocuMind AI is a multi-modal RAG system built with FastAPI, Weaviate, and Claude Sonnet for intelligent document question answering."],
        ["The embedding model used is sentence-transformers/all-MiniLM-L6-v2, loaded directly via HuggingFace transformers library."],
        ["Documents are ingested using Unstructured.io with hi_res strategy, which extracts text, tables, and layout information before chunking."],
        ["DocuMind implements hybrid search combining BM25 sparse retrieval and dense vector search, fused using Reciprocal Rank Fusion (RRF)."],
        ["The reranker used is BAAI/bge-reranker-v2-m3, a cross-encoder model that scores each retrieved chunk against the query for precise reranking."]
    ],
    "ground_truth": [
        "DocuMind AI is a production multi-modal RAG system for intelligent document querying.",
        "The embedding model is sentence-transformers/all-MiniLM-L6-v2.",
        "DocuMind uses Unstructured.io in hi_res mode for document ingestion.",
        "DocuMind uses BM25 and dense hybrid search with RRF fusion.",
        "DocuMind uses BAAI/bge-reranker-v2-m3 as the cross-encoder reranker."
    ]
}

dataset = Dataset.from_dict(sample_data)
print(f"Dataset ready: {len(dataset)} samples")

Dataset ready: 5 samples


In [12]:
from ragas.embeddings import BaseRagasEmbeddings
from sentence_transformers import SentenceTransformer
from typing import List
import asyncio

class CustomEmbeddings(BaseRagasEmbeddings):
    def __init__(self):
        self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode(text).tolist()
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()

    async def aembed_query(self, text: str) -> List[float]:
        return self.embed_query(text)

    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.embed_documents(texts)

embeddings = CustomEmbeddings()
print("Embeddings ready")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4284.95it/s]


Embeddings ready


In [13]:
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
    llm=llm,
    embeddings=embeddings
)

print(result)

Evaluating: 100%|██████████| 20/20 [00:10<00:00,  1.94it/s]


{'faithfulness': 0.6000, 'answer_relevancy': 0.8328, 'context_recall': 0.8000, 'context_precision': 1.0000}


In [15]:
import json

eval_results = {
    "model": "claude-sonnet-4-6",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "dataset_size": len(dataset),
    "scores": {
        "faithfulness": result["faithfulness"],
        "answer_relevancy": result["answer_relevancy"],
        "context_recall": result["context_recall"],
        "context_precision": result["context_precision"]
    }
}

print(json.dumps(eval_results, indent=2))

{
  "model": "claude-sonnet-4-6",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "dataset_size": 5,
  "scores": {
    "faithfulness": [
      0.5,
      0.5,
      0.6666666666666666,
      1.0,
      0.3333333333333333
    ],
    "answer_relevancy": [
      0.9999999999999177,
      0.9544361277407715,
      0.7243699598106156,
      0.8367889730811676,
      0.6485060543143969
    ],
    "context_recall": [
      0.0,
      1.0,
      1.0,
      1.0,
      1.0
    ],
    "context_precision": [
      0.9999999999,
      0.9999999999,
      0.9999999999,
      0.9999999999,
      0.9999999999
    ]
  }
}


In [16]:
import numpy as np

eval_results = {
    "model": "claude-sonnet-4-6",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "dataset_size": len(dataset),
    "scores": {
        "faithfulness": round(float(np.mean(result["faithfulness"])), 4),
        "answer_relevancy": round(float(np.mean(result["answer_relevancy"])), 4),
        "context_recall": round(float(np.mean(result["context_recall"])), 4),
        "context_precision": round(float(np.mean(result["context_precision"])), 4)
    }
}

print(json.dumps(eval_results, indent=2))

{
  "model": "claude-sonnet-4-6",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "dataset_size": 5,
  "scores": {
    "faithfulness": 0.6,
    "answer_relevancy": 0.8328,
    "context_recall": 0.8,
    "context_precision": 1.0
  }
}
